# 03 — Family splits, observed replay, and demonstrations

**Status: UNEXECUTED FIRST DRAFT.** Purpose: construct validated, provenance-preserving transition data through the public ARC interface. Inputs: configured train/dev/heldout game IDs and user-provided public assets; optional genuine demonstrations. Outputs: `data/splits.json`, `data/replay/train.jsonl`, collection manifest, committed episode files, and `reports/data_integrity.json`. Expected runtime: a bounded default collection of up to three episodes / 64 actions each / 120 seconds, plus validation; profile required. Run All creates missing exploration data only from configured **train** families; it never downloads competition data or reads game source.

Checkpoint directory: `checkpoints/<run_id>/` for model training. Collection checkpoints live beside replay in `<stem>_collection_manifest.json`; each completed or intentionally budget-truncated episode is an atomic committed unit. Resume: rerun this notebook with unchanged configuration/source and replay path. Completed episode files are reused, an uncommitted interrupted episode is marked abandoned and restarted with a **new** ID, and its staging records never enter training. Public ARC hidden state is not exact-resumable. Replay-buffer and sampler state enable exact **optimization** continuation later, not identical external environment continuation.

Configure family-disjoint IDs before collection. An official game's versions share its prefix before `-`; no family may cross train/dev/heldout. This is a public-development split, not evidence on the hidden competition set. The [official toolkit](https://docs.arcprize.org/toolkit/arc_agi) drives environment interactions; the [Kaggle data page](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/data) was not downloaded or accepted, and no submission-runner equivalence is claimed.

## Load the explicit environment/runtime interfaces

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([2], globals())

## Enforce family separation
A saved split manifest is immutable for an existing campaign. Change run/data paths deliberately when changing splits. Empty training splits stop collection clearly; no automatic catalogue-to-training assignment is made.

In [ ]:
import collections
import math
import os
import random
import tempfile
import torch

class SplitRegistry:
    def __init__(self, train_games=None, dev_games=None, heldout_games=None):
        if isinstance(train_games, dict):
            raw = train_games.get('data', train_games)
            raw = raw.get('splits', raw)
            train_games, dev_games, heldout_games = (raw.get('train_games', raw.get('train', [])),
                raw.get('dev_games', raw.get('dev', [])), raw.get('heldout_games', raw.get('heldout', [])))
        self.splits = {'train': list(train_games or []), 'dev': list(dev_games or []), 'heldout': list(heldout_games or [])}
        self.families = {}
        for split, games in self.splits.items():
            if len(games) != len(set(games)):
                raise ValueError(f'Duplicate game IDs in {split}.')
            for game in games:
                if not isinstance(game, str) or not game.strip():
                    raise ValueError('Every game ID must be a nonempty string.')
                family = game.split('-', 1)[0]
                previous = self.families.setdefault(family, split)
                if previous != split:
                    raise ValueError(f'Game-family leakage: {family} occurs in {previous} and {split}.')

    @classmethod
    def from_config(cls, cfg):
        return cls(cfg)

    @classmethod
    def from_manifest(cls, path):
        manifest = json.loads(Path(path).read_text())
        if manifest.get('schema_version') != 1 or manifest.get('family_policy') != 'prefix-before-hyphen':
            raise ValueError('Unsupported split manifest schema or family policy.')
        return cls(manifest)

    def split_for(self, game_id):
        return self.families.get(str(game_id).split('-', 1)[0])

    def assert_game(self, game_id, split):
        if split not in self.splits:
            raise ValueError(f'Unknown split: {split}')
        actual = self.split_for(game_id)
        if actual != split:
            raise ValueError(f'Game {game_id} belongs to {actual!r}, not {split!r}; family separation is required.')
        return True

    def to_dict(self):
        return {'schema_version': 1, 'scope': 'public_developer',
                'family_policy': 'prefix-before-hyphen', 'splits': copy.deepcopy(self.splits)}

    def validate(self):
        return {'games': {split: len(games) for split, games in self.splits.items()},
                'families': len(self.families), 'family_overlap': False, 'scope': 'public_developer'}

    def save(self, path):
        path = Path(path)
        value = self.to_dict()
        if path.exists() and canonical_hash(json.loads(path.read_text())) != canonical_hash(value):
            raise ValueError('Existing split manifest differs; use a separate campaign/data path rather than overwrite it.')
        atomic_json(path, value)
        return value

## Validate observed transitions and teacher provenance
Rewards are explicitly labelled local training signals: completed-level delta, with no invented reward for imagined states. A teacher target is accepted only for `human_demonstration` or `expert_demonstration`, with a nonempty `provenance.teacher_id`. Exploration actions never become teacher labels. Imported demonstrations must already be in this schema and preserve their origin; this notebook does not relabel a random action as expert.

In [ ]:
TEACHER_SOURCES = frozenset({'human_demonstration', 'expert_demonstration'})
TRANSITION_REQUIRED = frozenset({'episode_id', 'step', 'game_id', 'split', 'observation', 'action',
    'next_observation', 'reward', 'done', 'available_actions', 'next_available_actions', 'source', 'provenance'})

def validate_transition(row, registry=None):
    if not isinstance(row, dict) or TRANSITION_REQUIRED.difference(row):
        raise ValueError(f'Missing transition fields: {sorted(TRANSITION_REQUIRED.difference(row if isinstance(row,dict) else {}))}')
    result = copy.deepcopy(row)
    if not isinstance(result['episode_id'], str) or not result['episode_id']:
        raise ValueError('episode_id must be a nonempty string.')
    if isinstance(result['step'], bool) or not isinstance(result['step'], int) or result['step'] < 0:
        raise ValueError('Transition step must be a nonnegative integer.')
    if result['split'] not in {'train', 'dev', 'heldout'} or not isinstance(result['game_id'], str) or not result['game_id']:
        raise ValueError('Transition requires a valid game_id and split.')
    if registry is not None:
        registry.assert_game(result['game_id'], result['split'])
    for key in ('observation', 'next_observation'):
        result[key] = validate_grid(result[key]).tolist()
    result['available_actions'] = validate_available_actions(result['available_actions'])
    result['next_available_actions'] = validate_available_actions(result['next_available_actions'])
    actual_action = ActionSpec.from_dict(result['action'])
    if actual_action.action not in result['available_actions'] and actual_action.action != 0:
        raise ValueError('Recorded action was not in the observed legal-action list.')
    result['action'] = actual_action.to_dict()
    if isinstance(result['reward'], bool) or not isinstance(result['reward'], (int, float)) or not math.isfinite(result['reward']):
        raise ValueError('Reward must be finite numeric data.')
    result['reward'] = float(result['reward'])
    if not isinstance(result['done'], bool):
        raise ValueError('done must be a boolean from the actual observed terminal state.')
    if 'truncated' in result and not isinstance(result['truncated'], bool):
        raise ValueError('truncated must be boolean and distinct from done.')
    provenance = result['provenance']
    if not isinstance(provenance, dict) or provenance.get('kind') != 'observed' or not provenance.get('interface'):
        raise ValueError('Replay requires observed public-interface provenance; imagined predictions are not experience.')
    if not isinstance(result['source'], str) or not result['source']:
        raise ValueError('source must identify the collection policy or demonstration origin.')
    if result.get('teacher_action') is not None:
        if result['source'] not in TEACHER_SOURCES or not isinstance(provenance.get('teacher_id'), str) or not provenance['teacher_id'].strip():
            raise ValueError('Teacher targets require explicit demonstration source and provenance.teacher_id.')
        teacher = ActionSpec.from_dict(result['teacher_action'])
        if teacher.action not in result['available_actions'] and teacher.action != 0:
            raise ValueError('Teacher action was not legal in the public observation.')
        result['teacher_action'] = teacher.to_dict()
    else:
        result.pop('teacher_action', None)
    return result

def load_transitions(path, split=None, registry=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing observed replay file: {path}')
    if split is not None and split not in {'train', 'dev', 'heldout'}:
        raise ValueError('Unknown requested split.')
    rows, episode_last, seen = [], {}, set()
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                row = validate_transition(json.loads(line), registry)
                key = (row['episode_id'], row['step'])
                if key in seen:
                    raise ValueError('Duplicate episode/step transition.')
                seen.add(key)
                previous = episode_last.get(row['episode_id'])
                if previous is None:
                    if row['step'] != 0 or row.get('episode_start') is not True:
                        raise ValueError('Every episode must begin at step 0 with episode_start=True.')
                else:
                    if previous['done'] or previous.get('truncated', False):
                        raise ValueError('An episode contains transitions after termination/truncation.')
                    if row['step'] != previous['step'] + 1 or row['game_id'] != previous['game_id'] or row['split'] != previous['split']:
                        raise ValueError('Episode steps/game/split are inconsistent.')
                    if row['observation'] != previous['next_observation'] or row['available_actions'] != previous['next_available_actions']:
                        raise ValueError('Adjacent transitions disagree about the observed board or actions.')
                episode_last[row['episode_id']] = row
                if split is None or row['split'] == split:
                    rows.append(row)
            except Exception as exc:
                raise ValueError(f'{path}:{line_number}: {exc}') from exc
    return rows

def _action_xy(action):
    return [2.0 * action['x'] / 63.0 - 1.0, 2.0 * action['y'] / 63.0 - 1.0] if action['action'] == 6 else [0.0, 0.0]

def transition_batch(rows, device):
    if not rows:
        raise ValueError('Cannot batch an empty replay sample.')
    rows = [validate_transition(row) for row in rows]
    actual = [row['action'] for row in rows]
    teachers = [row.get('teacher_action', row['action']) for row in rows]
    def tensor(values, dtype):
        return torch.as_tensor(values, dtype=dtype, device=device)
    masks = [[action == 0 or action in row['available_actions'] for action in range(8)] for row in rows]
    next_masks = [[action == 0 or action in row['next_available_actions'] for action in range(8)] for row in rows]
    return {'grids': tensor([row['observation'] for row in rows], torch.long),
            'actions': tensor([a['action'] for a in actual], torch.long),
            'xy': tensor([_action_xy(a) for a in actual], torch.float32),
            'next_grids': tensor([row['next_observation'] for row in rows], torch.long),
            'rewards': tensor([row['reward'] for row in rows], torch.float32),
            'dones': tensor([row['done'] for row in rows], torch.float32),
            'masks': tensor(masks, torch.bool), 'next_masks': tensor(next_masks, torch.bool),
            'teacher_mask': tensor(['teacher_action' in row for row in rows], torch.bool),
            'teacher_actions': tensor([a['action'] for a in teachers], torch.long),
            'teacher_xy': tensor([_action_xy(a) for a in teachers], torch.float32),
            'teacher_click_targets': tensor([[a['x'],a['y']] if a['action']==6 else [-1,-1] for a in teachers], torch.long),
            'click_targets': tensor([[a['x'],a['y']] if a['action']==6 else [-1,-1] for a in actual], torch.long),
            'truncated': tensor([row.get('truncated', False) for row in rows], torch.bool)}

## Stateful replay and deterministic sample positions
Save these states in the training checkpoint's `extra` payload. A replay sample is drawn only from supplied validated rows; no hidden environment state enters the buffer. `sample_transition_batch` uses notebook 01's checkpointed `StatefulSampler` so the next batch can be reproduced after an optimizer restart.

In [ ]:
def _nested_tuple(value):
    return tuple(_nested_tuple(item) for item in value) if isinstance(value, (tuple, list)) else value

class ReplayBuffer:
    def __init__(self, capacity, seed):
        self.capacity = int(capacity)
        if self.capacity <= 0:
            raise ValueError('Replay capacity must be positive.')
        self.rows = []
        self.position = 0
        self.total_added = 0
        self.rng = random.Random(int(seed))

    def __len__(self):
        return len(self.rows)

    def add(self, row):
        value = validate_transition(row)
        if len(self.rows) < self.capacity:
            self.rows.append(value)
        else:
            self.rows[self.position] = value
        self.position = (self.position + 1) % self.capacity
        self.total_added += 1

    def sample(self, count):
        count = int(count)
        if count <= 0 or count > len(self.rows):
            raise ValueError('Replay sample count must be positive and no larger than current buffer size.')
        return copy.deepcopy(self.rng.sample(self.rows, count))

    def state_dict(self):
        return {'schema_version': 1, 'capacity': self.capacity, 'rows': copy.deepcopy(self.rows),
                'position': self.position, 'total_added': self.total_added, 'rng': self.rng.getstate()}

    def load_state_dict(self, state):
        if state.get('schema_version') != 1 or int(state['capacity']) != self.capacity:
            raise ValueError('Replay state schema/capacity mismatch.')
        rows = [validate_transition(row) for row in state['rows']]
        if len(rows) > self.capacity or not 0 <= int(state['position']) < self.capacity:
            raise ValueError('Invalid replay state size or position.')
        self.rows = rows
        self.position = int(state['position'])
        self.total_added = int(state['total_added'])
        self.rng.setstate(_nested_tuple(state['rng']))

def sample_transition_batch(rows, sampler, batch_size, device):
    indices = sampler.next_indices(int(batch_size))
    batch = transition_batch([rows[int(index)] for index in indices], device)
    batch['sample_indices'] = torch.as_tensor(indices, dtype=torch.long, device=device)
    return batch

def _atomic_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + str(uuid.uuid4()) + '.tmp')
    try:
        with temporary.open('w', encoding='utf-8') as handle:
            for row in rows:
                handle.write(json.dumps(row, allow_nan=False, separators=(',', ':')) + '\n')
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()

class AtomicEpisodeWriter:
    def __init__(self, directory, episode_id):
        self.directory = Path(directory)
        self.episode_id = str(episode_id)
        self.staging = self.directory / 'staging' / self.episode_id
        self.staging.mkdir(parents=True, exist_ok=True)
        self.rows = []
        self.committed = False
        atomic_json(self.staging / 'status.json', {'status': 'collecting', 'episode_id': self.episode_id,
                    'exact_environment_resume_supported': False})

    def append(self, row):
        row = validate_transition(row)
        if self.committed or row['episode_id'] != self.episode_id or row['step'] != len(self.rows):
            raise ValueError('Episode writer requires consecutive rows for its own uncommitted episode.')
        atomic_json(self.staging / f"{row['step']:06d}.json", row)
        self.rows.append(row)

    def commit(self, outcome):
        if self.committed or not self.rows:
            raise RuntimeError('Cannot commit an empty or already committed episode.')
        if outcome not in {'terminal', 'budget_truncated'}:
            raise ValueError('Only finished or deliberately bounded episodes can be committed.')
        self.rows[-1]['truncated'] = outcome == 'budget_truncated' and not self.rows[-1]['done']
        destination = self.directory / 'episodes' / f'{self.episode_id}.jsonl'
        _atomic_jsonl(destination, self.rows)
        record = {'episode_id': self.episode_id, 'path': str(destination.resolve().relative_to(ROOT.resolve())),
                  'sha256': sha256_file(destination), 'transitions': len(self.rows), 'outcome': outcome,
                  'exact_environment_resume_supported': False}
        atomic_json(self.staging / 'status.json', dict(record, status='committed'))
        self.committed = True
        return record

    def abandon(self, reason):
        if not self.committed:
            atomic_json(self.staging / 'status.json', {'status': 'abandoned', 'episode_id': self.episode_id,
                        'reason': str(reason), 'transitions_observed': len(self.rows),
                        'exact_environment_resume_supported': False, 'included_in_training': False})

def make_transition(observation, action, next_observation, split, source, step, provenance=None, teacher_action=None):
    action = action if isinstance(action, ActionSpec) else ActionSpec.from_dict(action)
    row = {'episode_id': observation['episode_id'], 'step': int(step), 'game_id': observation['game_id'],
           'split': split, 'observation': validate_grid(observation['grid']).tolist(), 'action': action.to_dict(),
           'next_observation': validate_grid(next_observation['grid']).tolist(),
           'reward': float(next_observation['levels_completed'] - observation['levels_completed']),
           'done': terminal_observation(next_observation), 'available_actions': observation['available_actions'],
           'next_available_actions': next_observation['available_actions'], 'source': source,
           'episode_start': int(step) == 0,
           'provenance': dict(provenance or {}, kind='observed', interface='official-arc-agi-public-frame',
                              reward_definition='observed-levels-completed-delta', guid=observation.get('guid'))}
    if teacher_action is not None:
        row['teacher_action'] = teacher_action.to_dict() if isinstance(teacher_action,ActionSpec) else teacher_action
    return validate_transition(row)

## Bounded train-only collection with an episode journal
Run All collection is OFFLINE and at most ten minutes even if its arguments are changed. Each actual response is saved atomically to staging; only a committed episode enters replay. A crash after a real action has uncertain environment state: no automatic action retry is attempted. On restart the journal abandons its active episode and resumes the campaign between episodes with the next new episode ID. Source/configuration changes are rejected.

In [ ]:
def collect_public_rollouts(cfg, registry, output_path, episodes=3, max_episode_actions=64, max_wall_seconds=120):
    require_recovery_gate(root=ROOT)
    if str(cfg['environment'].get('operation_mode','OFFLINE')).upper() != 'OFFLINE':
        raise RuntimeError('Default data collection is OFFLINE only; remote evaluation is a separate explicit run.')
    episodes, max_episode_actions = int(episodes), int(max_episode_actions)
    max_wall_seconds = float(max_wall_seconds)
    if not 1 <= episodes <= 100 or not 1 <= max_episode_actions <= 256 or not 0 < max_wall_seconds <= 600:
        raise ValueError('Collection must be bounded to 1..100 episodes, 1..256 actions/episode, and <=600 seconds/call.')
    games = registry.splits['train']
    if not games:
        raise ValueError('Configure a nonempty train split before collecting public experience.')
    output_path = Path(output_path)
    directory = output_path.parent / (output_path.stem + '_collection')
    journal_path = output_path.parent / (output_path.stem + '_collection_manifest.json')
    identity = {'config_hash': canonical_hash(cfg), 'split_hash': canonical_hash(registry.to_dict()),
                'source_hash': canonical_hash(source_manifest(root=ROOT)),
                'episodes': episodes, 'max_episode_actions': max_episode_actions,
                'seed': int(cfg['train'].get('seed',42))}
    with RunLock(root=ROOT, run_id='collect-' + output_path.stem):
        if journal_path.exists():
            journal = json.loads(journal_path.read_text())
            if journal.get('identity') != identity:
                raise ValueError('Collection source/config/splits changed; choose a new replay path for the new campaign.')
        else:
            if output_path.exists() and output_path.stat().st_size:
                raise ValueError('Existing replay has no matching collection journal; validate/import it rather than overwrite it.')
            journal = {'schema_version':1, 'identity':identity, 'committed':[], 'abandoned':[], 'active_episode':None,
                       'exact_environment_resume_supported':False}
        if journal.get('active_episode'):
            abandoned = journal['active_episode']
            journal['abandoned'].append(dict(abandoned, reason='process-interrupted-hidden-state-not-restored'))
            status_path = directory / 'staging' / abandoned['episode_id'] / 'status.json'
            atomic_json(status_path, {'status':'abandoned','episode_id':abandoned['episode_id'],
                                     'reason':'process-interrupted','included_in_training':False})
            journal['active_episode'] = None
        atomic_json(journal_path, journal)
        deadline = time.monotonic() + max_wall_seconds
        while len(journal['committed']) < episodes and time.monotonic() < deadline:
            episode_index = len(journal['committed'])
            game_id = games[episode_index % len(games)]
            registry.assert_game(game_id, 'train')
            episode_cfg = copy.deepcopy(cfg)
            episode_cfg['environment']['seed'] = int(cfg['environment'].get('seed',42)) + episode_index
            episode_cfg['environment']['max_actions'] = max_episode_actions
            rng = random.Random(identity['seed'] + episode_index)
            writer = None
            try:
                with ARCSession(episode_cfg, operation_mode='OFFLINE') as session:
                    observation = session.reset(game_id)
                    writer = AtomicEpisodeWriter(directory, observation['episode_id'])
                    journal['active_episode'] = {'episode_id':observation['episode_id'], 'game_id':observation['game_id'],
                                                 'episode_index':episode_index}
                    atomic_json(journal_path,journal)
                    for step in range(max_episode_actions):
                        choices = [action for action in observation['available_actions'] if action != 0]
                        if terminal_observation(observation) or not choices or time.monotonic() >= deadline:
                            break
                        action_id = rng.choice(choices)
                        action = ActionSpec(action_id,rng.randrange(64),rng.randrange(64)) if action_id==6 else ActionSpec(action_id)
                        next_observation = session.step(action)
                        row = make_transition(observation,action,next_observation,'train','random_rollout',step,
                            provenance={'policy':'seeded-uniform-public-action','seed':identity['seed']+episode_index,
                                        'toolkit_version':session.toolkit_version})
                        registry.assert_game(row['game_id'],'train')
                        writer.append(row)
                        observation = next_observation
                        if row['done']:
                            break
                    if writer.rows:
                        outcome = 'terminal' if terminal_observation(observation) else 'budget_truncated'
                        committed = writer.commit(outcome)
                        journal['committed'].append(committed)
                    else:
                        writer.abandon('No actual non-reset transition returned.')
                        journal['abandoned'].append(dict(journal['active_episode'],reason='empty-episode'))
                        raise RuntimeError('No actionable public transition was available for collection.')
                    journal['active_episode'] = None
                    atomic_json(journal_path,journal)
            except BaseException as exc:
                if writer is not None:
                    writer.abandon(type(exc).__name__)
                # Retain active_episode in journal so restart treats its hidden state as abandoned.
                atomic_json(journal_path,journal)
                raise
        rows = []
        for record in journal['committed']:
            path = ROOT / record['path']
            if sha256_file(path) != record['sha256']:
                raise ValueError('Committed episode hash mismatch; refusing changed experience.')
            rows.extend(load_transitions(path,split='train',registry=registry))
        _atomic_jsonl(output_path,rows)
        journal['materialized_replay_sha256'] = sha256_file(output_path)
        journal['materialized_transitions'] = len(rows)
        journal['status'] = 'completed' if len(journal['committed']) == episodes else 'bounded-call-complete'
        atomic_json(journal_path,journal)
        return journal

## Record or import genuine demonstrations

`human_demonstration_widget` provides a real public board, selectable legal actions and explicit ACTION6 coordinates. Press **Start** to open one configured training game, make the moves yourself, then **Save episode**. Each executed move is its own teacher target with your stated teacher ID. The recorder is bounded to 64 actions and ten minutes; it saves at every move to staging and commits a complete or deliberately truncated episode. An interrupted episode is not restored or claimed exact. This UI is defined here and displayed by Run All only when teacher data are absent; no actions occur until its buttons are pressed.

`import_demonstrations(source_path, destination, registry)` imports existing JSONL in the schema above. It requires explicit demonstration source, `teacher_action`, and `provenance.teacher_id` on every row, checks family separation and duplicate identity, and rejects random rollouts instead of relabelling them.

In [ ]:
def import_demonstrations(source_path, destination, registry):
    source_path, destination = Path(source_path), Path(destination)
    with RunLock(root=ROOT,run_id='demonstrations-' + destination.stem):
        incoming = load_transitions(source_path, split='train', registry=registry)
        if not incoming:
            raise ValueError('Demonstration import contains no training transitions.')
        for row in incoming:
            if row['source'] not in TEACHER_SOURCES or 'teacher_action' not in row:
                raise ValueError('Only explicit teacher-provenance demonstration records may be imported.')
        existing = load_transitions(destination, split='train', registry=registry) if destination.exists() else []
        indexed = {(row['episode_id'],row['step']): row for row in existing}
        for row in incoming:
            key = (row['episode_id'],row['step'])
            if key in indexed and canonical_hash(indexed[key]) != canonical_hash(row):
                raise ValueError('Conflicting demonstration episode/step identity.')
            indexed[key] = row
        merged = list(indexed.values())
        _atomic_jsonl(destination,merged)
        return {'path':str(destination),'transitions':len(merged),'imported_transitions':len(incoming),
                'sha256':sha256_file(destination),'source':'explicit-demonstrations'}
class HumanDemonstrationRecorder:
    def __init__(self,cfg,registry,destination,max_actions=64,max_wall_seconds=600):
        import ipywidgets as widgets
        self.cfg,self.registry,self.destination = copy.deepcopy(cfg),registry,Path(destination)
        self.max_actions,self.max_wall_seconds = int(max_actions),float(max_wall_seconds)
        if not 1 <= self.max_actions <= 256 or not 0 < self.max_wall_seconds <= 600:
            raise ValueError('Human recording is bounded to <=256 actions and <=600 seconds.')
        if str(cfg['environment'].get('operation_mode','OFFLINE')).upper() != 'OFFLINE':
            raise RuntimeError('Human demonstration UI records public OFFLINE training games only.')
        if not registry.splits['train']:
            raise ValueError('Configure training game families before recording.')
        self.session,self.writer,self.observation,self.started = None,None,None,None
        self.game = widgets.Dropdown(options=registry.splits['train'],description='Train game:')
        self.teacher = widgets.Text(description='Teacher ID:',placeholder='Your stable non-secret identifier')
        self.action = widgets.Dropdown(options=[],description='Action:')
        self.x,self.y = widgets.IntSlider(min=0,max=63,description='x:'),widgets.IntSlider(min=0,max=63,description='y:')
        self.start_button = widgets.Button(description='Start',button_style='primary')
        self.step_button = widgets.Button(description='Execute teacher action',disabled=True)
        self.save_button = widgets.Button(description='Save episode',disabled=True)
        self.output = widgets.Output()
        self.start_button.on_click(self._start)
        self.step_button.on_click(self._step)
        self.save_button.on_click(self._finish_callback)
        self.widget = widgets.VBox([widgets.HBox([self.game,self.teacher]),self.start_button,
                                   widgets.HBox([self.action,self.x,self.y]),widgets.HBox([self.step_button,self.save_button]),self.output])

    def _draw(self,message=''):
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap
        from IPython.display import display
        self.output.clear_output(wait=True)
        with self.output:
            if message:
                print(message)
            if self.observation is not None:
                figure,axis=plt.subplots(figsize=(5,5))
                axis.imshow(self.observation['grid'],vmin=0,vmax=15,cmap=ListedColormap(('#FFFFFF','#CCCCCC','#999999','#666666','#333333','#000000','#E53AA3','#FF7BCC','#F93C31','#1E93FF','#88D8F1','#FFDC00','#FF851B','#921231','#4FCC30','#A356D6')),interpolation='nearest')
                axis.set_title(f"Observed {self.observation['game_id']} | {self.observation['state']} | levels={self.observation['levels_completed']}")
                axis.set_xlabel('x');axis.set_ylabel('y')
                display(figure);plt.close(figure)
        if self.observation is not None:
            choices=[a for a in self.observation['available_actions'] if a!=0]
            self.action.options=[(f'ACTION{a}',a) for a in choices]
            self.step_button.disabled=self.session is None or not choices or terminal_observation(self.observation)

    def _start(self,_):
        try:
            if self.session is not None:
                raise RuntimeError('Save the current episode before starting another.')
            if not self.teacher.value.strip():
                raise ValueError('Enter a teacher ID before recording; it is stored as provenance.')
            require_recovery_gate(root=ROOT)
            self.registry.assert_game(self.game.value,'train')
            episode_cfg=copy.deepcopy(self.cfg)
            episode_cfg['environment']['max_actions']=self.max_actions
            self.session=ARCSession(episode_cfg,operation_mode='OFFLINE')
            self.observation=self.session.reset(self.game.value)
            self.writer=AtomicEpisodeWriter(self.destination.parent/'human_collection',self.observation['episode_id'])
            self.started=time.monotonic()
            self.teacher_id=self.teacher.value.strip()
            self.start_button.disabled=True;self.save_button.disabled=False
            self._draw('Real human demonstration. Coordinates matter only for ACTION6.')
        except Exception as exc:
            if self.session is not None:
                self.session.close();self.session=None
            self._draw(f'Cannot start: {type(exc).__name__}: {exc}')

    def _step(self,_):
        try:
            if self.session is None or self.writer is None:
                raise RuntimeError('Start a demonstration first.')
            if time.monotonic()-self.started >= self.max_wall_seconds or len(self.writer.rows)>=self.max_actions:
                self.finish();return
            chosen=int(self.action.value)
            action=ActionSpec(chosen,int(self.x.value),int(self.y.value)) if chosen==6 else ActionSpec(chosen)
            next_observation=self.session.step(action)
            row=make_transition(self.observation,action,next_observation,'train','human_demonstration',len(self.writer.rows),
                provenance={'teacher_id':self.teacher_id,'toolkit_version':self.session.toolkit_version,'policy':'human-controlled-ui'},
                teacher_action=action)
            self.writer.append(row)
            self.observation=next_observation
            self._draw(f'Teacher action recorded atomically. Moves: {len(self.writer.rows)}')
            if row['done'] or len(self.writer.rows)>=self.max_actions:
                self.finish()
        except Exception as exc:
            if self.writer is not None:
                self.writer.abandon(type(exc).__name__)
            if self.session is not None:
                self.session.close();self.session=None
            self.step_button.disabled=True;self.save_button.disabled=True;self.start_button.disabled=False
            self._draw(f'Episode abandoned; no exact environment resume: {type(exc).__name__}: {exc}')

    def _finish_callback(self,_):
        try:
            self.finish()
        except Exception as exc:
            self._draw(f'Save failed: {type(exc).__name__}: {exc}')

    def finish(self):
        if self.session is None or self.writer is None:
            raise RuntimeError('No active human episode.')
        try:
            if not self.writer.rows:
                self.writer.abandon('Empty human demonstration.')
                self._draw('No action was recorded; no teacher data added.')
                return None
            outcome='terminal' if terminal_observation(self.observation) else 'budget_truncated'
            record=self.writer.commit(outcome)
            result=import_demonstrations(ROOT/record['path'],self.destination,self.registry)
            self._draw(f"Saved {record['transitions']} genuine teacher transitions to {self.destination}.")
            return result
        finally:
            self.session.close();self.session=None
            self.start_button.disabled=False;self.step_button.disabled=True;self.save_button.disabled=True

def human_demonstration_widget(cfg,registry,destination=None):
    from IPython.display import display
    recorder=HumanDemonstrationRecorder(cfg,registry,destination or ROOT/cfg['data']['expert_path'])
    display(recorder.widget)
    return recorder

## Later Run All: collect missing replay, validate all configured data
This cell performs the bounded collection described above only when replay is missing or a previous journal has fewer than three committed episodes. Existing user demonstrations are loaded and validated, never generated. Empty teacher coverage is reported honestly; world-model training may proceed on experience, while imitation has no labelled targets until genuine demonstrations exist.

In [ ]:
cfg = load_config()
require_recovery_gate(root=ROOT)
registry = SplitRegistry(cfg)
if not registry.splits['train']:
    raise RuntimeError('Configure data.train_games/dev_games/heldout_games with disjoint game families in config/ahr_v2_xl.json first.')
registry.save(ROOT / cfg['data']['split_manifest'])
replay_path = ROOT / cfg['data']['replay_path']
journal_path = replay_path.parent / (replay_path.stem + '_collection_manifest.json')
should_collect = not replay_path.exists()
if journal_path.exists():
    collection_state = json.loads(journal_path.read_text())
    should_collect = len(collection_state.get('committed',[])) < 3 or bool(collection_state.get('active_episode'))
if should_collect:
    collection_report = collect_public_rollouts(cfg,registry,replay_path,episodes=3,max_episode_actions=64,max_wall_seconds=120)
    print({key:collection_report[key] for key in ('status','materialized_transitions','exact_environment_resume_supported')})
replay_rows = load_transitions(replay_path,split='train',registry=registry)
expert_path = ROOT / cfg['data']['expert_path']
demonstrations = load_transitions(expert_path,split='train',registry=registry) if expert_path.exists() else []
if any(row['source'] not in TEACHER_SOURCES or 'teacher_action' not in row for row in demonstrations):
    raise ValueError('Demonstration file must contain explicit verified teacher targets for every row.')
report = dict(registry.validate(), status='passed', replay_transitions=len(replay_rows),
              demonstration_transitions=len(demonstrations), teacher_targets=sum('teacher_action' in row for row in replay_rows+demonstrations),
              replay_sha256=sha256_file(replay_path), expert_sha256=sha256_file(expert_path) if expert_path.exists() else None,
              exact_environment_resume_supported=False, metric_scope='public-development-data-integrity-not-performance')
atomic_json(ROOT/'reports'/'data_integrity.json',report)
print(report)
if not demonstrations:
    demonstration_recorder = human_demonstration_widget(cfg,registry,expert_path)
    print('Supervised imitation requires genuine demonstrations. Use this recorder, save episodes, then rerun the validation cell before notebook 10.')

## Training handoff
`load_transitions(path, split='train', registry=registry)` supplies checked rows. `transition_batch` keeps real executed actions for dynamics; its separate teacher tensors are valid only under `teacher_mask`. Save `ReplayBuffer.state_dict()` and the notebook 01 sampler's state with every resumable training checkpoint. Budgets and termination are different: a time/action limit sets `truncated`, never invents `done` or a win. A public family held out locally remains a public family, and repeated evaluation should be disclosed.